# Running FORCE Level 2 and Time Series Analysis

### Purpose

The [Earth Observation Application Package (EOAP)](https://docs.ogc.org/bp/20-089r1.html) and surrounding tools implemented in the [`apex-force-openeo`](https://github.com/bcdev/apex-force-openeo) repository make it possible to use the [FORCE](https://force-eo.readthedocs.io/en/latest/) EO processing engine on the [Copernicus Data Space Ecosystem (CDSE)](https://dataspace.copernicus.eu/) using the [openEO](https://openeo.dataspace.copernicus.eu/) API.
This notebook uses `pystac_client` to make a STAC query to determine input products. This is a valid alternative to the `query_stac` process in OpenEO, especially until it is supported on the production environment.

> This notebook serves to showcase how to combine FORCE level 2 and FORCE Time Series Analysis (TSA) on CDSE openEO. For a more detailed discussion on how to run FORCE level 2, have a look at the [FORCE level 2 examples](../run-level2-and-download).

Please note that this is neither a tutorial for FORCE nor for openEO, please have a look at the Furhter Reading section below, if you are unfamiliar with either. The basic openEO client operations will be explained, FORCE will be used without further explanation. Please refer to the [FORCE documentation](https://force-eo.readthedocs.io/en/latest) for the [level 2 processing system](https://force-eo.readthedocs.io/en/latest/components/lower-level/level2/) and the [Time Series Analysis (TSA) module](https://force-eo.readthedocs.io/en/latest/components/higher-level/tsa/index.html#time-series-analysis) and to learn more.


### Prerequisites

In order to run this notebook, you need to have a CDSE account and sufficient openEO credits to run the jobs. Please note that CDSE provides a contigent of free credits each month, so you should be able to run the notebook if you haven't used up your credits with no additional cost.

> At the time of writing (2026-04-20) the query process `query_stac` requires an account for the staging deployment of the openEO backend. However, it should be available on the production deployment soon.

To learn how run FORCE level 2 (without TSA) check out the [FORCE level 2 examples](../run-level2-and-download).
There, the process to run FORCE level 2 is explained in more detail.

### Content

1. Connect to OpenEO backend
1. Make a query to the CDSE STAC catalog to determine input products
1. Run the FORCE level 2 processing system (l2ps) on CDSE using the openEO API
1. Run the FORCE Time Series Analysis (TSA) module on CDSE using the openEO API and the results from level 2 processing
1. Download processed products
1. Visualize the results

### Citation and Acknowledgement

When using the FORCE processing engine, please make sure to properly [cite and acknowledge](https://force-eo.readthedocs.io/en/latest/policy/citation.html)
the software and its related [scientific publications](https://force-eo.readthedocs.io/en/latest/refs.html#refs) .
The implementation as EOAP was performed in the context of ESA's [APEx](https://apex.esa.int/) initiative.

### Further reading

**APEx**

1. [Project website](https://apex.esa.int/)
2. [Documentation Portal](https://esa-apex.github.io/apex_documentation/)

**FORCE**

1. [Documentation](https://force-eo.readthedocs.io/en/latest/index.html)
1. [Level 2 processing system](https://force-eo.readthedocs.io/en/latest/components/lower-level/level2/)
1. [Citation and acknowledgement](https://force-eo.readthedocs.io/en/latest/policy/citation.html)


**openEO**

1. [openEO on CDSE](https://documentation.dataspace.copernicus.eu/APIs/openEO/openEO.html)
1. [Credit Usage on CDSE](https://documentation.dataspace.copernicus.eu/APIs/openEO/credit_usage.html)
1. [openEO project website](https://openeo.org/)
1. [openEO Python Client documentation](https://open-eo.github.io/openeo-python-client/)

### Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import force_tsa_example_utils as utils
import openeo
import pystac
import pystac_client

### Area of Interest

##### Ancona
w,s,e,n = 13.2, 43.439, 13.688, 43.75
spatial_extent = { "west": w, "south": s, "east": e, "north": n}
aoi_name = "Ancona"
temporal_extent=["2026-02-01", "2026-02-20"]

In [3]:
##### Modena

w,s,e,n = 10.386, 44.437, 11.423, 44.973
spatial_extent = { "west": w, "south": s, "east": e, "north": n}
aoi_name = "Modena"
temporal_extent=["2026-04-17", "2026-04-18"]

utils.plot_area_of_interest(
    w=w,
    s=s,
    e=e,
    n=n,
    large_context=4,
    figsize=(10, 6),
    title=f"Spatial Extent: {aoi_name}"
)

### Connect to the OpenEO backend

In [4]:
backend_url = "openeo-staging.dataspace.copernicus.eu"
#backend_url = "openeo.dataspace.copernicus.eu"
connection = openeo.connect(backend_url).authenticate_oidc()

Authenticated using refresh token.


### Query the CDSE STAC catalog

In [5]:
L1C_COLLECTION_URL = "https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-l1c"

client = pystac_client.Client.open("https://stac.dataspace.copernicus.eu/v1")
search = client.search(
    datetime=temporal_extent,
    collections=["sentinel-2-l1c"],
    bbox=[w, s, e, n]
)
item_collection = search.item_collection()

print("Found items:")
for item in search.items():
    print(item.id)


Found items:
S2C_MSIL1C_20260417T101021_N0512_R022_T32TPQ_20260417T134842
S2C_MSIL1C_20260417T101021_N0512_R022_T32TNQ_20260417T134842


In [6]:
# This is currently necessary to pass a longer list of inputs to FORCE level 2
catalog = utils.transform_item_collection_to_catalog_with_links(item_collection)
catalog

<Catalog id=item-collection-catalog>

### Running FORCE level 2 (l2ps)

We need to tell FORCE on which products to run. The processor will automatically download the necessary files.
As usual in openEO, we can chain processes. In this case, we can connect the query directly to FORCE level2, without evaluating it.
Note that we use the openEO process graph `query_pg` and not the query results as a parameter.

#### Note:

**You will be able to use a dedicated process `force_level2` instead of `run_cwl_to_stac` which will make it unnecessary to explicitly pass the CWL document in the future.
Until this process is available on the CDSE openEO backend, `run_cwl_to_stac` can be used.**

#### openEO

You can track the progress of your job by logging into the openEO web interface 

Alternatively, in a separate Python shell / notebook, you can run the following snippet to get access to the logs as a Python list.
You can find out the job id from the next cell below.

Please be aware that the output from CWL jobs is not optimized for openEO logs, so the logs may look quite messy and fragmented. 
However, they can be a valuable source of information to debug issues.

```Python
connection = ... # connect to the same backend
l2_job = connection.job(<job-id>)
l2_logs = l2_job.logs()
print("\n".join(l2_logs[:20]))
```

In [7]:
cwl_text = Path("force-level2.cwl").read_text()
#cwl_text = Path("../run-level2-and-download/force-level2.cwl").read_text()

nproc = 2
context = dict(
    stac_document=catalog.to_dict(),
    nproc=nproc,
)

stac_resource = openeo.rest.stac_resource.StacResource(
    graph=openeo.internal.graph_building.PGNode(
        process_id="run_cwl_to_stac",
        arguments={
            "cwl": cwl_text,
            "context": context,
        }
    ),
    connection=connection,
)
l2_job = stac_resource.create_job(title=f"FORCE level 2 - {aoi_name} - {nproc}")
print(f"Job id: '{l2_job.job_id}'")
l2_job.start()

Job id: 'j-2605051459074e1e9398f739c5a5290f'


<BatchJob job_id='j-2605051459074e1e9398f739c5a5290f'>

## Extracting TSA parameters from the Level 2 results

### Catalog and Item URLs

To run the TSA module on the results produced by FORCE level 2, we must pass it the STAC catalog that was produced by the FORCE level 2 process.

> We are currently still waiting for an openEO feature to obtain the catalog url directly from the job results.
> Because of this, we use the workaround shown here `extract_catalog_url_from_job_logs` to get the correct URL.
> In the future, you can use the following code to access the catalog URL instead. **The code will already run without error (on the staging environment) but produce an incorrect URL**:

```Python
catalog_url = next(l for l in l2_results.get_metadata()["links"] if l["rel"] == "child")["href"]
```

The STAC Catalog contains a single item representing a single [FORCE datacube](https://force-eo.readthedocs.io/en/latest/howto/datacube.html). 

In [8]:
#l2_job = connection.job("j-2605051156324d16904625e330b429ff")

In [9]:
l2_logs = l2_job.logs()
l2_results = l2_job.get_results()

In [10]:
l2_catalog_url = utils.extract_catalog_url_from_job_logs(l2_logs)
l2_catalog = pystac.read_file(l2_catalog_url)
l2_item = next(iter(l2_catalog.get_items()))
l2_item

StopIteration: 

### Tiles

When processing in the cloud, we cannot just inspect the processed datacube in the local file system. Instead, we can inspect the STAC catalog produced by the FORCE level 2 process to gather information about the resulting cube.
The processed tiles (in the sense of the [FORCE datacube model](https://force-eo.readthedocs.io/en/latest/howto/datacube.html)) can be determined from the assets present in the STAC Item.
This information can be useful to set the `x_tile_range` and `y_tile_range` parameters for higher level processing, for example:

In [ ]:
tiles = set([k.split(".")[1] for k in l2_item.assets.keys() if k.endswith("BOA")])
tile_x = set(int(tile[1:5]) for tile in tiles)
tile_y = set(int(tile[7:]) for tile in tiles)
tile_x, tile_y

## Running Force Time Series Analysis (TSA) on a subset

After level 2 processing, we can use the [FORCE TSA module](https://force-eo.readthedocs.io/en/latest/components/higher-level/tsa/) to generate pixel wise time series statistics (for example). The level 2 datacube is passed to the TSA module as a URL pointing to the STAC catalog (or Item) extracted in the last step.

You may want to test the configuration for the TSA module on a small subset of the tiles, to make sure you are getting the correct results. You can run FORCE TSA multiple times on the same input with different configurations by passing the same level 2 catalog to the process.

In [ ]:
# process parameters
cwl_tsa = Path("force-tsa.cwl").read_text()

context = dict(
    stac_url=l2_catalog.get_self_href(),
    date_range=temporal_extent,
    x_tile_range=[min(tile_x), min(tile_x)],
    y_tile_range=[min(tile_y), min(tile_y)],
    stm=["AVG"],
    output_stm=True,
)

job_options={
  "executor-memory": "10G",
  "python-memory": "9G",
}


stac_resource = openeo.rest.stac_resource.StacResource(
    graph=openeo.internal.graph_building.PGNode(
        process_id="run_cwl_to_stac",
        arguments={
            "cwl": cwl_tsa,
            "context": context,
        },
    ),
    connection=connection,
)
tsa_job = stac_resource.create_job(
    title=f"FORCE TSA - {aoi_name}",
    job_options=job_options,
)
tsa_job.start()

In [ ]:
#tsa_job = connection.job("j-2604241127384bbcb86b37799dd0ff57")

In [ ]:
assets_path = Path("force-tsa-small")

In [ ]:
tsa_results = tsa_job.get_results()
tsa_results.download_files(assets_path)

In [ ]:
X, Y = "0032", "0030"
asset = "20260201-20260220_001-365_HL_TSA_SEN2L_NDVI_STM"
tiff_path = assets_path / f"X{X}_Y{Y}" / f"{asset}.tif"
utils.plot_asset_ndvi(tiff_path);

## Running FORCE Time Series Analysis on the full cube

After verifying that results look as intended, we can run TSA again on the full datacube.

In [ ]:
# update process parameters
context = context | dict(
    x_tile_range=[min(tile_x), max(tile_x)],
    y_tile_range=[min(tile_y), max(tile_y)],
    stm=["AVG", "MAX"],
)

stac_resource_2 = openeo.rest.stac_resource.StacResource(
    graph=openeo.internal.graph_building.PGNode(
        process_id="run_cwl_to_stac",
        arguments={
            "cwl": cwl_tsa,
            "context": context,
        },
    ),
    connection=connection,
)
tsa_job_2 = stac_resource_2.create_job(
    title=f"FORCE TSA - {aoi_name} - full",
    job_options=job_options,
)
tsa_job_2.start()

In [ ]:
tsa_job_2 = connection.job("j-260424144114463fb97426637ca3cc5b")
tsa_logs_2 = tsa_job_2.logs()

In [ ]:
[l.message for l in tsa_logs_2[-20:]]

In [ ]:
assets_path_2 = Path("force-tsa-full")
tsa_results_2 = tsa_job_2.get_results()
tsa_results_2.download_files(assets_path_2)

In [ ]:
X, Y = "0032", "0030"
asset = "20260201-20260220_001-365_HL_TSA_SEN2L_NDVI_STM"
tiff_path = assets_path / f"X{X}_Y{Y}" / f"{asset}.tif"
utils.plot_asset_ndvi(tiff_path)